# Repair C: Qwen7B History + Internal Candidates

This notebook **does not regenerate candidates or reload Qwen** when the completed C predictions already exist. It repairs the malformed emotion outputs using a robust parser, recomputes all metrics on the canonical 1,592-point IEMOCAP test split, and writes corrected results under `/workspace/utterance/C_history_plus_internal_candidates_qwen7b_corrected/`.

Place these in `/workspace` before running:

- `IEMOCAP_features.pkl`
- either the previous C result ZIP, `predictions_partial.jsonl`, or the extracted C result folder


In [ ]:
from pathlib import Path
from dataclasses import dataclass
from collections import Counter
import json, pickle, re, zipfile, shutil

DATA_PATH = Path('/workspace/IEMOCAP_features.pkl')
OUTPUT_DIR = Path('/workspace/utterance/C_history_plus_internal_candidates_qwen7b_corrected')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f'Missing dataset: {DATA_PATH}'
print('Dataset:', DATA_PATH)
print('Output:', OUTPUT_DIR)


In [ ]:
# Locate or extract the previous C predictions automatically.
workspace = Path('/workspace')
extract_root = workspace / 'c_previous_extracted'

zip_candidates = list(workspace.glob('*C_history_plus_internal_candidates_qwen7b*.zip')) + list(workspace.glob('*internal_candidates*.zip'))
if zip_candidates:
    extract_root.mkdir(parents=True, exist_ok=True)
    for z in zip_candidates:
        print('Extracting:', z)
        with zipfile.ZipFile(z) as archive:
            archive.extractall(extract_root)

search_roots = [workspace, extract_root]
jsonl_candidates = []
for root in search_roots:
    if root.exists():
        jsonl_candidates.extend(root.rglob('predictions_partial.jsonl'))
        jsonl_candidates.extend(root.rglob('*predictions*.jsonl'))

# Prefer a path containing the C experiment name.
jsonl_candidates = sorted(set(jsonl_candidates), key=lambda p: (
    'C_history_plus_internal_candidates_qwen7b' not in str(p),
    len(str(p))
))

if not jsonl_candidates:
    raise FileNotFoundError(
        'Could not find the previous C predictions JSONL. Place the C ZIP or predictions_partial.jsonl in /workspace.'
    )

PREDICTIONS_PATH = jsonl_candidates[0]
print('Using predictions:', PREDICTIONS_PATH)


In [ ]:
EMOTION_LABELS = ['neutral', 'frustration', 'sadness', 'anger', 'excited', 'happiness']
LABEL2ID = {x: i for i, x in enumerate(EMOTION_LABELS)}
NUMERIC_TO_LABEL = {0: 'happiness', 1: 'sadness', 2: 'neutral', 3: 'anger', 4: 'excited', 5: 'frustration'}
NORMALIZE = {
    'neutral': 'neutral', 'neu': 'neutral',
    'frustration': 'frustration', 'frustrated': 'frustration', 'fru': 'frustration',
    'sadness': 'sadness', 'sad': 'sadness',
    'anger': 'anger', 'angry': 'anger', 'ang': 'anger',
    'excited': 'excited', 'excitement': 'excited', 'exc': 'excited',
    'happiness': 'happiness', 'happy': 'happiness', 'hap': 'happiness',
}

@dataclass
class Sample:
    dialogue_id: str
    history: list
    history_speakers: list
    history_emotions: list
    target_speaker: str
    target_emotion: str
    target_emotion_id: int

def normalize_label(x):
    if isinstance(x, int):
        return NUMERIC_TO_LABEL.get(x)
    try:
        if hasattr(x, 'item'):
            value = x.item()
            if isinstance(value, int):
                return NUMERIC_TO_LABEL.get(value)
    except Exception:
        pass
    return NORMALIZE.get(str(x).strip().lower())

def speaker_label(x):
    if isinstance(x, (list, tuple)):
        return 'A' if max(range(len(x)), key=lambda i: x[i]) == 0 else 'B'
    try:
        if hasattr(x, 'shape') and len(x.shape) > 0:
            values = x.tolist()
            return 'A' if max(range(len(values)), key=lambda i: values[i]) == 0 else 'B'
    except Exception:
        pass
    return str(x)

def carve_val(train_vids, n_val=20):
    ordered = sorted(train_vids)
    val = set(ordered[-n_val:])
    return set(ordered[:-n_val]), val

def emit_samples(vid, utterances, speakers, labels):
    speakers = [speaker_label(s) for s in speakers]
    labels = [normalize_label(x) for x in labels]
    rows = []
    for t in range(1, len(utterances)):
        if t >= len(labels) or labels[t] is None:
            continue
        rows.append(Sample(
            dialogue_id=f'{vid}_t{t}', history=list(utterances[:t]),
            history_speakers=list(speakers[:t]), history_emotions=list(labels[:t]),
            target_speaker=speakers[t], target_emotion=labels[t],
            target_emotion_id=LABEL2ID[labels[t]],
        ))
    return rows

def load_canonical_iemocap(path):
    with open(path, 'rb') as f:
        raw = pickle.load(f, encoding='latin1')
    if isinstance(raw, (list, tuple)) and len(raw) >= 9:
        video_speakers, video_labels, video_sentence = raw[1], raw[2], raw[6]
        train_vids, test_vids = list(raw[7]), set(raw[8])
    elif isinstance(raw, dict) and ('videoSentence' in raw or 'trainVid' in raw):
        video_speakers = raw['videoSpeakers']; video_labels = raw['videoLabels']
        video_sentence = raw['videoSentence']; train_vids = list(raw['trainVid']); test_vids = set(raw['testVid'])
    else:
        raise ValueError('Expected the standard DialogueRNN-style IEMOCAP pickle.')
    train_set, dev_set = carve_val(train_vids, 20)
    splits = {'train': [], 'dev': [], 'test': []}
    for vid, utts in video_sentence.items():
        if vid in test_vids: split = 'test'
        elif vid in dev_set: split = 'dev'
        elif vid in train_set: split = 'train'
        else: continue
        splits[split].extend(emit_samples(vid, utts, video_speakers[vid], video_labels[vid]))
    return splits

splits = load_canonical_iemocap(DATA_PATH)
samples = splits['test']
assert len(samples) == 1592, f'Expected 1592 test samples, got {len(samples)}'
print({k: len(v) for k, v in splits.items()})


In [ ]:
def parse_emotion_fixed(text):
    """Recover valid IEMOCAP labels from malformed outputs without mapping unsupported emotions.

    Handles outputs such as:
      frustrationone_label
      <sadness>one_valid_label</emotion>
      <emotion>sadnessone_valid_label</emotion>
      neutralone_valid_label
    Unsupported labels such as concern, sorrow, skepticism remain parse failures.
    """
    low = str(text or '').lower().strip()

    aliases = [
        ('frustration', 'frustration'), ('frustrated', 'frustration'),
        ('happiness', 'happiness'), ('happy', 'happiness'),
        ('excitement', 'excited'), ('excited', 'excited'),
        ('sadness', 'sadness'), ('sad', 'sadness'),
        ('anger', 'anger'), ('angry', 'anger'),
        ('neutral', 'neutral'),
    ]

    hits = []
    for alias, normalized in aliases:
        # Deliberately allow suffixes such as 'sadnessone_label'.
        for match in re.finditer(re.escape(alias), low):
            hits.append((match.start(), len(alias), normalized))

    if not hits:
        return None

    # Prefer the latest mention; for ties, prefer the longer alias.
    return max(hits, key=lambda x: (x[0], x[1]))[2]

# Required regression tests.
tests = {
    'frustrationone_label': 'frustration',
    '<sadness>one_valid_label</emotion>': 'sadness',
    '<emotion>sadnessone_valid_label</emotion>': 'sadness',
    'neutralone_valid_label': 'neutral',
    '<emotion>concern</emotion>': None,
    '<sorrow>one_valid_label</emotion>': None,
}
for raw, expected in tests.items():
    actual = parse_emotion_fixed(raw)
    assert actual == expected, (raw, expected, actual)
print('PARSER TESTS PASSED')


In [ ]:
def load_jsonl(path):
    with Path(path).open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

records = load_jsonl(PREDICTIONS_PATH)
by_id = {}
for row in records:
    by_id[row['dialogue_id']] = row

missing = [s.dialogue_id for s in samples if s.dialogue_id not in by_id]
assert not missing, f'Missing {len(missing)} predictions; first missing: {missing[:5]}'
assert len(by_id) >= 1592, f'Expected at least 1592 unique predictions, got {len(by_id)}'

corrected = []
for s in samples:
    row = dict(by_id[s.dialogue_id])
    old_pred = row.get('predicted_emotion')
    new_pred = parse_emotion_fixed(row.get('raw_output', ''))
    row['predicted_emotion_original'] = old_pred
    row['predicted_emotion'] = new_pred
    row['parser_repaired'] = old_pred != new_pred
    corrected.append(row)

print('Unique predictions:', len(by_id))
print('Changed by repaired parser:', sum(r['parser_repaired'] for r in corrected))
print('Remaining parse failures:', sum(r['predicted_emotion'] is None for r in corrected))


In [ ]:
def same_speaker_shift(s):
    for spk, emo in zip(reversed(s.history_speakers), reversed(s.history_emotions)):
        if spk == s.target_speaker and emo is not None:
            return s.target_emotion != emo
    return None

def f1_for_label(y_true, y_pred, label):
    tp = sum(t == label and p == label for t, p in zip(y_true, y_pred))
    fp = sum(t != label and p == label for t, p in zip(y_true, y_pred))
    fn = sum(t == label and p != label for t, p in zip(y_true, y_pred))
    denom = 2 * tp + fp + fn
    return 0.0 if denom == 0 else 2 * tp / denom

def metric_block(y_true, y_pred):
    labels = list(range(6))
    per_class = {EMOTION_LABELS[l]: f1_for_label(y_true, y_pred, l) for l in labels}
    supports = Counter(y_true)
    n = len(y_true)
    weighted = sum(per_class[EMOTION_LABELS[l]] * supports.get(l, 0) for l in labels) / n if n else 0.0
    macro = sum(per_class.values()) / len(labels)
    accuracy = sum(t == p for t, p in zip(y_true, y_pred)) / n if n else 0.0
    return {'weighted_f1': weighted, 'macro_f1': macro, 'accuracy': accuracy, 'per_class_f1': per_class}

PARSE_FAIL_ID = -1
yt = [s.target_emotion_id for s in samples]
yp = [LABEL2ID.get(r.get('predicted_emotion'), PARSE_FAIL_ID) for r in corrected]

es_idx = [i for i, s in enumerate(samples) if same_speaker_shift(s) is True]
ns_idx = [i for i, s in enumerate(samples) if same_speaker_shift(s) is False]

all_metrics = metric_block(yt, yp)
metrics = {
    'title': 'C_history_plus_internal_candidates_qwen7b_corrected_parser',
    'n': len(samples),
    **all_metrics,
    'parse_failures': sum(p == PARSE_FAIL_ID for p in yp),
    'parser_repairs': sum(r['parser_repaired'] for r in corrected),
    'es_n': len(es_idx),
    'es_metrics': metric_block([yt[i] for i in es_idx], [yp[i] for i in es_idx]),
    'no_shift_n': len(ns_idx),
    'no_shift_metrics': metric_block([yt[i] for i in ns_idx], [yp[i] for i in ns_idx]),
    'undefined_n': len(samples) - len(es_idx) - len(ns_idx),
}
print(json.dumps(metrics, indent=2))


In [ ]:
corrected_jsonl = OUTPUT_DIR / 'predictions_corrected.jsonl'
with corrected_jsonl.open('w', encoding='utf-8') as f:
    for row in corrected:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

results_path = OUTPUT_DIR / 'results_corrected.json'
results_path.write_text(json.dumps({'metrics': metrics, 'predictions': corrected}, indent=2, ensure_ascii=False), encoding='utf-8')

readme = OUTPUT_DIR / 'README.txt'
readme.write_text(
    'C: history + four internal candidates -> emotion\n'
    'This folder contains metrics recomputed from the original completed generation run using a repaired parser.\n'
    'No candidates were regenerated and no model was reloaded.\n',
    encoding='utf-8'
)

zip_path = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)
print('Saved corrected JSONL:', corrected_jsonl)
print('Saved corrected results:', results_path)
print('ZIP:', zip_path)
